In [ ]:
import sys
import os
import re
import joblib
import numpy as np
import gensim
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from slovene_pipeline.word2vec_api import Word2VecAPI, maybe_load_emoji2vec
from slovene_pipeline.features import build_tfidf, build_w2v_mean, combine_features
from irony_translation.LLMSarcasmTranslator import LLMSarcasmTranslator
from irony_translation.SarcasmTranslator import T5SarcasmTranslator

In [ ]:
import pandas as pd
from transformers import pipeline

MODEL_PATH = os.path.join(project_root, 'slovene_pipeline', 'subtaskA_notebooks', 'finalized_model_og.sav')

if os.path.exists(MODEL_PATH):
    irony_classifier = joblib.load(MODEL_PATH)
    if isinstance(irony_classifier, dict) and "vectorizer" in irony_classifier:
        vectorizer = irony_classifier["vectorizer"]
        clf = irony_classifier["classifier"]
    else:
        clf = irony_classifier
        vectorizer = None 
else:
    print(f"Model not found at {MODEL_PATH}")

kv = gensim.models.fasttext.load_facebook_model('../w2v-slo/all-token-prelim.ft.sg.bin').wv
w2v = Word2VecAPI(kv)
emoji_model = maybe_load_emoji2vec(
    os.path.join(project_root, 'emoji2vec-master', 'pre-trained', 'emoji2vec.bin'),
    binary=True
)

# Option 1: LLM Translator (Groq/OpenAI base)
API_KEY = "api_key_here" 
llm_translator = LLMSarcasmTranslator(api_key=API_KEY)

# Option 2: Local HuggingFace Translator (e.g., mT5 trained on dataset)
local_translator = T5SarcasmTranslator(model_name="csebuetnlp/mT5_multilingual_XLSum")
# Uncomment next line to load local weights if trained locally
local_translator.load_model(os.path.join(project_root, "irony_translation", "from_medium_dataset", "nllb_200_distilled_600M"))

# Choose which one to pass to the pipeline
translator = llm_translator 
#translator = local_translator

try:
    sentiment_nlp = pipeline("sentiment-analysis", model="cardiffnlp/twitter-xlm-roberta-base-sentiment", tokenizer="cardiffnlp/twitter-xlm-roberta-base-sentiment")
except Exception as e:
    print("Warning: HuggingFace sentiment pipeline could not be loaded:", e)
    sentiment_nlp = None

try:
    df_emoji = pd.read_csv(os.path.join(project_root, 'extra_resources', 'Emoji_Sentiment_Data_v1.0.csv'))
    df_emoji = df_emoji[['Emoji', 'Negative', 'Neutral', 'Positive']]
    idx2lb = {0: -1, 1: 0, 2: 1}
    emoji_sentimens = {}
    for val in df_emoji.values:
         emoji_sentimens[val[0]] = idx2lb[np.argmax(np.array(val[1:]))]
except Exception as e:
    print("Warning: Emoji sentiment dataset could not be loaded:", e)
    emoji_sentimens = {}

In [ ]:
def chunkIt(seq, n):
    avg = len(seq) / float(n)
    out = []
    last = 0.0
    while last < len(seq):
        out.append(seq[int(last):int(last + avg)])
        last += avg
    return out

def mock_sentiment_score(text):
    if isinstance(text, list): text = ' '.join(text)
    text = (text or '').strip()
    if not text: return 2
    if sentiment_nlp is None: return 2
    try:
        pred = sentiment_nlp(text[:512])[0]
        label = str(pred.get('label', '')).lower()
        conf = float(pred.get('score', 0.0))
        if label in {'label_0', 'negative'}: return 0 if conf >= 0.85 else 1
        elif label in {'label_1', 'neutral'}: return 2
        else: return 4 if conf >= 0.85 else 3
    except: return 2

def extractHashtag(tweet):
    t = tweet.split(' ')
    text = []
    hashtagText = []
    oneHashtag = []
    flag = 0
    for w in t:
        if w == "<hashtag>":
            flag = 1; continue
        if flag == 1:
            if w == "</hashtag>":
                hashtagText.append(' '.join(oneHashtag))
                oneHashtag = []
                flag = 0
            else:
                oneHashtag.append(w)
        else:
            text.append(w)
    return ' '.join(text), hashtagText

def extract_enhanced_features(tweet_text):
    """
    Extract exactly 58 features mimicking feature_generator_TaskA_og.
    """
    from ekphrasis.utils.nlp import polarity
    import re
    import numpy as np

    words = tweet_text.split()
    if not words:
        return np.zeros(58).tolist()

    # 1. Structural Intensity (3 features)
    chunks_words = chunkIt(words, 2)
    part1_str = ' '.join(chunks_words[0]) if len(chunks_words) > 0 else ""
    part2_str = ' '.join(chunks_words[1]) if len(chunks_words) > 1 else ""

    output1 = mock_sentiment_score(part1_str)
    output2 = mock_sentiment_score(part2_str)
    leftIntensity = rightIntensity = polarityDiff = 0
    
    if output1 in [0, 4]: leftIntensity = 1
    if output2 in [0, 4]: rightIntensity = 1
    if (output1 > 2 and output2 < 2) or (output1 < 2 and output2 > 2):
        polarityDiff = 1
    feats_1 = [leftIntensity, rightIntensity, polarityDiff]

    # 2. Contrast feature (1 feature)
    emojis_in_tweet = [emoji_sentimens[e] for e in re.findall(r'[\U0001f600-\U0001f650]', tweet_text) if e in emoji_sentimens]
    txt_part, htag_parts = extractHashtag(tweet_text)
    
    txt_sentiment = mock_sentiment_score(txt_part)
    htag_sentiment = [mock_sentiment_score(h) for h in htag_parts]
    
    hset = set(htag_sentiment)
    eset = set(emojis_in_tweet)
    
    contrast_val = 0
    if (txt_sentiment in {2, 3, 4}) and (hset & {0, 1}): contrast_val = 1
    elif (txt_sentiment in {0, 1}) and (hset & {3, 4}): contrast_val = 1
    elif (txt_sentiment in {2, 3, 4}) and (-1 in eset): contrast_val = 1
    elif (txt_sentiment in {0, 1}) and (1 in eset): contrast_val = 1
    elif {-1, 1}.issubset(eset): contrast_val = 1
    elif {0, 4}.issubset(hset) or {0, 3}.issubset(hset) or {1, 4}.issubset(hset): contrast_val = 1
    elif (hset & {0, 1}) and (1 in eset): contrast_val = 1
    elif (hset & {3, 4}) and (-1 in eset): contrast_val = 1
    contrast_feat = [contrast_val]

    # 3. Ekphrasis Tags (24 tags * 2 chunks = 48 features)
    tags = ['<allcaps>', '<annoyed>', '<censored>', '<date>', '<elongated>', '<emphasis>', '<happy>',
            '<hashtag>', '<heart>', '<kiss>', '<laugh>', '<money>', '<number>', '<percent>', '<phone>',
            '<repeated>', '<sad>', '<shocking>', '<surprise>', '<time>', '<tong>', '<url>', '<user>',
            '<wink>']
            
    ekphrasis_feats = []
    for chunk in chunks_words:
        for tag in tags:
            ekphrasis_feats.append(sum(1 for t in chunk if t == tag))
            
    while len(ekphrasis_feats) < 48:
        ekphrasis_feats.append(0)


    pol1 = polarity(part1_str) if part1_str else None
    pol2 = polarity(part2_str) if part2_str else None
    
    p1 = np.array(pol1[1]) if (pol1 and len(pol1)>1) else np.zeros(3)
    p2 = np.array(pol2[1]) if (pol2 and len(pol2)>1) else np.zeros(3)
    
    exact_feats = feats_1 + contrast_feat + ekphrasis_feats + p1.tolist() + p2.tolist()
    
    if len(exact_feats) < 58:
        exact_feats += [0.0] * (58 - len(exact_feats))
    elif len(exact_feats) > 58:
        exact_feats = exact_feats[:58]

    return exact_feats

In [ ]:
def is_ironic(sentence: str, clf_model=None, w2v_model=None, emoji_model=None):
    if clf_model is None:
        return False
    
    try:
        # combining all features (from words2vec, emoji2vec and extract_enhanced_features())
        w2v_feat = build_w2v_mean([sentence], w2v_model, emoji_model)[0]
        enhanced_feat = np.array(extract_enhanced_features(sentence))
        x = np.hstack([w2v_feat, enhanced_feat]).reshape(1, -1)
        
        preds = clf_model.predict(x)
        return bool(preds[0])
    except Exception as e:
        print(f"Error during sentence prediction: {e}")
        return False

In [ ]:
def translate_ironic_sentence(sentence: str, translator_model) -> str:
    try:
        if hasattr(translator_model, 'zero_shot'):
            # Used by LLMSarcasmTranslator
            non_ironic = translator_model.zero_shot(sentence)
        elif hasattr(translator_model, 'generate'):
            # Used by T5SarcasmTranslator
            non_ironic = translator_model.generate(sentence)
        else:
            return sentence
            
        return non_ironic.strip()
    except Exception as e:
        print(f"Translation failed: {e}")
        return sentence

In [ ]:
def process_text_pipeline(input_text: str, clf_model=None, vectorizer_model=None, w2v_model=None, emoji_model=None, translator_model=None):
    is_ironic_text = is_ironic(
        input_text, 
        clf_model=clf_model, 
        w2v_model=w2v_model, 
        emoji_model=emoji_model
    )
    
    final_text = input_text
    transformed = False
    
    if is_ironic_text and translator_model is not None:
        final_text = translate_ironic_sentence(input_text, translator_model)
        transformed = True
        
    result = {
        "original_text": input_text,
        "final_text": final_text,
        "is_ironic": is_ironic_text,
        "transformed": transformed
    }
    
    return result

In [ ]:
print("clf:", clf)
print("vectorizer:", vectorizer)
print("w2v:", w2v)
print("emoji_model:", emoji_model)
print("translator:", translator)

Example usage:

In [ ]:
sample_texts = [
    "Oh, kako obožujem dve uri čakanja v koloni. Najboljši način za začetek jutra! 🚗🕰️ #zastoji #ljubljana #sreča",
    "To, da mi telefon crkne pri 2 %, ravno ko potrebujem navigacijo, je brez dvoma moja najljubša funkcija. 📱🔋 #tehnologija #popolno",
    "Uau, še en deževen vikend. Saj sem bil res že pošteno utrujen od preveč sonca letos. 🌧️☔ #vreme #slovenija #komajčakam",
    "Danes sem imel čudovit pohod na Šmarno goro! Razgled je na koncu vedno poplačan. 🏔️☀️ #šmarnagora #hribi #narava"
]
CLF = globals().get('clf', None)
VEC = globals().get('vectorizer', None)
W2V = globals().get('w2v', None)
EMOJI = globals().get('emoji_model', None)
TRANS = globals().get('translator', None)
for text in sample_texts:

    
    if CLF is None:
        def mock_is_ironic(sentence, *args, **kwargs):
            return "odličen" in sentence.lower() and "dežuje" in sentence.lower() or "seveda" in sentence.lower()
        
        original_is_ironic = is_ironic
        is_ironic = mock_is_ironic
    
    output = process_text_pipeline(text, clf_model=CLF, vectorizer_model=VEC, w2v_model=W2V, emoji_model=EMOJI, translator_model=TRANS)
    
    import json
    print(json.dumps(output, indent=2, ensure_ascii=False))
    
    if CLF is None:
         is_ironic = original_is_ironic

In [ ]:
import pandas as pd
# IMPORTANT tu je dejanski experiment
def run_experiment_on_file(input_csv_path: str, output_csv_path: str, ironic_csv_path: str = None):
    df = pd.read_csv(input_csv_path)
    
    CLF = globals().get('clf', None)
    VEC = globals().get('vectorizer', None)
    W2V = globals().get('w2v', None)
    EMOJI = globals().get('emoji_model', None)
    TRANS = globals().get('translator', None)
    
    results = []
    ironic_for_rerun = []
    
    for index, row in df.iterrows():
        text = str(row['text'])
        true_label = row.get('true_label', '')
        
        output = process_text_pipeline(
            text, 
            clf_model=CLF, 
            vectorizer_model=VEC, 
            w2v_model=W2V, 
            emoji_model=EMOJI, 
            translator_model=TRANS
        )
        
        results.append({
            'input_text': text,
            'true_label': true_label,
            'predicted_ironic': output.get('is_ironic', False),
            'output_text': output.get('final_text', text),
        })
        
        if output.get('is_ironic', False):
            ironic_for_rerun.append({
                'text': output.get('final_text', text),  # translated output becomes the new input
                'true_label': False,
            })
        
    out_df = pd.DataFrame(results)
    out_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
    print(f"Eksperiment zaključen. Podatki shranjeni v: {output_csv_path}")

    if ironic_for_rerun:
        ironic_path = ironic_csv_path or output_csv_path.replace('.csv', '_ironic_rerun.csv')
        ironic_df = pd.DataFrame(ironic_for_rerun)
        ironic_df.to_csv(ironic_path, index=False, encoding='utf-8-sig')
        print(f"Ironični primeri za ponovni zagon shranjeni v: {ironic_path} ({len(ironic_for_rerun)} vrstic)")
    else:
        print("Ni ironičnih primerov za ponovni zagon.")

# First run - full dataset, detects & translates ironic sentences
run_experiment_on_file("experiment_input.csv", "rezultati_eksperimenta_normal.csv", ironic_csv_path="za_ponovni_zagon.csv")

# Second run - only the translated sentences, check if they're now detected as non-ironic
run_experiment_on_file("za_ponovni_zagon.csv", "rezultati_ponovni_zagon.csv")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

def plot_confusion_matrix(csv_path: str):
    df = pd.read_csv(csv_path)

    y_true = df['true_label'].astype(int)

    y_pred = df['predicted_ironic'].map({
        True: 1,
        False: 0,
        'True': 1,
        'False': 0
    }).astype(int)

    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(6, 5))

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Ne ironično (0)', 'Ironično (1)']
    )

    disp.plot(ax=ax, colorbar=False, cmap='Blues')

    # Custom axis labels
    ax.set_xlabel('Napoved modela')
    ax.set_ylabel('Pravilna oznaka')

    ax.set_title(
        'Matrika zmede - klasifikacija transformacije na podlagi LLM',
        fontsize=13,
        pad=12
    )

    plt.tight_layout()
    plt.show()

    print(classification_report(
        y_true,
        y_pred,
        target_names=['Ne ironično', 'Ironično']
    ))

# usage
plot_confusion_matrix("rezultati_ponovni_zagon.csv")